In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 24 — O JULGAMENTO FINAL: ABRINDO O COFRE

> "Você pode adiar o julgamento, mas não pode evitá-lo. E quando ele chega, não há segunda chamada — só a verdade, nua, sobre o que você construiu."
> — Anotação no diário do Protagonista, na véspera

Eu me lembrava do voto que fizera no Capítulo 4. "Uma linha na areia que não será cruzada." Naquele dia, tranquei 120 pacientes dentro de um cofre e prometi esquecê-los. Por meses, cumpri a promessa. Cada gráfico da análise exploratória, cada *feature* que criei, cada modelo que testei, cada hiperparâmetro que afinei, cada limiar que escolhi — **tudo** foi decidido olhando apenas o conjunto de treino. O cofre permaneceu fechado, uma cápsula do tempo com o mundo real lá dentro.

Agora o modelo estava pronto. O comitê de especialistas, calibrado e explicável. Eu tinha estimativas — a validação aninhada me prometera algo em torno de 96,5% em dados novos. Mas estimativas são promessas. Eu nunca havia visto meu modelo diante de um paciente que ele nunca tinha encontrado. Até este instante.

O que me apertava o peito não era a curiosidade. Era a irreversibilidade. Eu podia abrir o cofre **uma única vez**. Se o resultado fosse ruim, eu não poderia voltar e reajustar nada — fazer isso transformaria o teste em mais um dado de treino, e a honestidade de toda a jornada iria por água abaixo. Era um exame sem retomada. E, no fundo daquele cofre, havia um rosto: os 120 pacientes eram o futuro que o OncoClassify 1.0 enfrentou e falhou. Eram, cada um deles, uma Helena em potencial. A pergunta que eu adiara por um livro inteiro finalmente exigia resposta: *desta vez, o sistema os veria?*

Respirei fundo. E digitei o comando.

## Por Que o Cofre se Abre Uma Só Vez

O conjunto de teste tem um único propósito: dar uma estimativa **imparcial** de como o modelo se comportará em dados que nunca viu. Essa imparcialidade é frágil — ela sobrevive a exatamente **um** uso. No instante em que você olha o resultado no teste e, com base nele, muda qualquer coisa (o modelo, o *threshold*, uma *feature*), o teste vira treino disfarçado e sua estimativa volta a ser otimista. Foi assim que o OncoClassify 1.0 se enganou.

Por isso o ritual é rígido: treinamos o modelo final **apenas no treino**, fazemos as previsões no teste **uma vez**, lemos o veredito — e o aceitamos, seja ele qual for. Note que treinamos no `X_train` (não nos 600 registros): a avaliação precisa ser feita por um modelo que jamais tocou nenhum dos pacientes do cofre.

#### Código 24.1: O Modelo Final Diante do Desconhecido


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from oncoclassify_utils import X_train, y_train, X_test, y_test   # o cofre entra em cena

# O comitê final (Capítulo 22), treinado SÓ no treino.
log = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(solver="liblinear", random_state=42))])
svm = Pipeline([("sc", StandardScaler()), ("svm", SVC(kernel="rbf", C=10, gamma=0.01, probability=True, random_state=42))])
gb  = GradientBoostingClassifier(n_estimators=100, random_state=42)
modelo_final = VotingClassifier([("lr", log), ("svm", svm), ("gb", gb)], voting="soft").fit(X_train, y_train)

# --- A abertura do cofre: UMA previsão, sem volta ---
y_pred = modelo_final.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Maligno", "Benigno"], digits=3))
print("Matriz de confusão [linhas=real, colunas=previsto]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

              precision    recall  f1-score   support

     Maligno      1.000     0.955     0.977        44
     Benigno      0.974     1.000     0.987        76

    accuracy                          0.983       120
   macro avg      0.987     0.977     0.982       120
weighted avg      0.984     0.983     0.983       120

Matriz de confusão [linhas=real, colunas=previsto]:
[[42  2]
 [ 0 76]]


![O cofre aberto — matriz de confusão no teste](media/figura_24_1.png)

O cofre está aberto. E o veredito é sólido:

**Acurácia: 98,3%** em 120 pacientes que o modelo **nunca havia visto**. Guarde este número e compare com o que prometemos: a validação aninhada (Capítulo 21) estimou **96,5%**; o teste real entregou **98,3%**. A estimativa honesta **se sustentou** — na verdade, foi até um pouco conservadora. Toda a disciplina do cofre valeu: nossa metodologia não estava se enganando.

**Dos 44 tumores malignos do cofre, o modelo encontrou 42.** A precisão maligna foi de **100%** — a coluna "Previsto: Maligno" não tem um único falso positivo: quando o modelo disse "câncer", ele estava certo todas as vezes. E os erros? **Dois falsos negativos** (a célula em vermelho). Dois — contra os 212 que o modelo de base cego deixaria passar, contra a falha silenciosa que custou Helena.

## O Último Gesto: O Threshold da Cautela

Dois falsos negativos ainda são dois pacientes. Podemos fazer melhor — e temos a ferramenta desde o Capítulo 16. Escolhemos o *threshold* de custo mínimo **no treino** (jamais no teste) e o aplicamos ao cofre **uma única vez**, honrando a mesma disciplina.

#### Código 24.2: Aplicando a Cautela ao Cofre


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from oncoclassify_utils import CUSTO_FN, CUSTO_FP

# 1) escolher o threshold de custo mínimo usando SÓ o treino (via CV)
p_treino = cross_val_predict(modelo_final, X_train, y_train, cv=5, method="predict_proba")[:, 0]
ymal_treino = (y_train == 0).astype(int).values
faixas = np.linspace(0.05, 0.95, 181)
def custo(th, p, ymal):
    prev = (p >= th).astype(int)
    fn = int(((prev == 0) & (ymal == 1)).sum()); fp = int(((prev == 1) & (ymal == 0)).sum())
    return CUSTO_FN * fn + CUSTO_FP * fp
t_otimo = faixas[int(np.argmin([custo(t, p_treino, ymal_treino) for t in faixas]))]

# 2) aplicar esse threshold ao COFRE, uma vez
p_teste = modelo_final.predict_proba(X_test)[:, 0]
prev_mal = (p_teste >= t_otimo).astype(int)
ymal_teste = (y_test == 0).astype(int).values
fn = int(((prev_mal == 0) & (ymal_teste == 1)).sum())
fp = int(((prev_mal == 1) & (ymal_teste == 0)).sum())
tp = int(((prev_mal == 1) & (ymal_teste == 1)).sum())

print(f"Threshold (escolhido no treino): {t_otimo:.2f}")
print(f"No cofre -> Recall Maligno: {tp/(tp+fn):.4f} | Falsos Negativos: {fn} | Falsos Positivos: {fp}")

Threshold (escolhido no treino): 0.05
No cofre -> Recall Maligno: 1.0000 | Falsos Negativos: 0 | Falsos Positivos: 10


> **⚠️ ATENÇÃO — O momento que responde à carta**
>
> Leia o número de novo: **Recall Maligno 1,0000. Falsos Negativos: 0.**
>
> Dos **44 pacientes com câncer que o modelo nunca tinha visto, ele encontrou todos os 44.** Nenhum passou despercebido. O preço foram **10 falsos positivos** — dez pacientes que farão um exame a mais e receberão, no fim, a notícia de que estão bem. É a troca que fizemos conscientemente lá no Capítulo 3, e que o custo clínico justificou no Capítulo 16: preferimos a ansiedade de um exame extra ao silêncio de um diagnóstico perdido.
>
> Foi exatamente esse silêncio que custou a Helena. Aqui, no conjunto que representa os pacientes futuros, ele não existiu. **Nenhuma Helena passaria despercebida.** Este é o número pelo qual o livro inteiro foi construído.

Uma última observação, sóbria: o cofre agora está **gasto**. Usamos o `X_test` — uma vez, como manda o ritual — e não podemos mais usá-lo para decidir nada. Se, a partir de agora, quiséssemos comparar uma nova ideia, precisaríamos de dados frescos. O teste cumpriu seu papel único: nos deu uma medida honesta e irrepetível da verdade.

> **📌 CHECKLIST DO CAPÍTULO 24**
>
> ✅ Entende por que o conjunto de teste se usa **uma única vez**, e por que reusá-lo o contamina.
>
> ✅ Sabe que o modelo avaliado deve ser treinado **só no treino** (não nos dados completos).
>
> ✅ Executou o Código 24.1 e obteve o veredito imparcial: **98,3%** de acurácia, **42 de 44** cânceres detectados, **zero** falsos positivos.
>
> ✅ Confirmou que o teste (**98,3%**) sustentou a estimativa da validação aninhada (**96,5%**) — a metodologia era honesta.
>
> ✅ Aplicou o *threshold* da cautela (escolhido no treino) e levou os Falsos Negativos a **zero** no cofre.
>
> **⚠️ CRÍTICO:** A avaliação final é irreversível. Depois dela, não se ajusta mais o modelo com base no teste. Se o resultado for ruim, isso é informação honesta — não um convite para "espiar e corrigir".

Fiquei olhando a tela por um longo tempo. Zero. Nos dados que o modelo nunca vira, nenhum câncer perdido. Eu não havia construído um sistema perfeito — dez pessoas saudáveis receberiam um susto, e isso não é nada. Mas eu havia construído um sistema **honesto**, e um sistema **cauteloso**, que errava para o lado certo. O cofre, que por meses foi uma promessa e uma disciplina, cumpriu sua função e se esvaziou. Sua última palavra foi a única que eu precisava ouvir.

Só agora, sabendo — não estimando, **sabendo** — que o modelo funcionava diante do desconhecido, eu tinha o direito de dar o próximo passo. Tirar o OncoClassify 2.0 do meu computador e colocá-lo onde ele poderia, de fato, ajudar alguém. Era hora de pensar em produção.

**PARTE VIII — PRODUÇÃO E ÉTICA**
